# NB_Sales_Quality
Contrôle qualité et chargement de l'export quotidien des ventes.

Appelé par le pipeline PL_Sales_Express avec le paramètre `load_date`.
Lit le fichier déposé en Files/raw, sépare les lignes valides des lignes rejetées,
écrit les deux dans des tables Delta, et renvoie un bilan JSON au pipeline.

In [ ]:
load_date = "2026-10-01"

In [ ]:
import json
from pyspark.sql import functions as F

source = f"Files/raw/sales/load_date={load_date}/sales.csv"
df = (spark.read.option("header", "true").csv(source)
      .withColumn("order_datetime", F.to_timestamp("order_datetime"))
      .withColumn("quantity",       F.col("quantity").cast("int"))
      .withColumn("unit_price",     F.col("unit_price").cast("double"))
      .withColumn("discount_pct",   F.col("discount_pct").cast("int"))
      .withColumn("line_amount",    F.col("line_amount").cast("double"))
      .withColumn("load_date",      F.lit(load_date)))
rows_read = df.count()

In [ ]:
# Règles de qualité : la première qui échoue donne la raison du rejet
df = df.withColumn("reject_reason",
     F.when(F.col("customer_id").isNull() | (F.trim("customer_id") == ""), "missing_customer")
      .when(F.col("quantity") <= 0,   "quantity_not_positive")
      .when(F.col("unit_price") <= 0, "unit_price_not_positive"))

valid    = df.filter(F.col("reject_reason").isNull()).drop("reject_reason")
rejected = df.filter(F.col("reject_reason").isNotNull())

In [ ]:
# Écriture idempotente : relancer la même date remplace la même date, sans doublon
(valid.write.format("delta").mode("overwrite")
      .option("replaceWhere", f"load_date = '{load_date}'").saveAsTable("sales"))
(rejected.write.format("delta").mode("overwrite")
      .option("replaceWhere", f"load_date = '{load_date}'").saveAsTable("sales_rejects"))

In [ ]:
# Bilan renvoyé au pipeline
report = {
    "load_date":     load_date,
    "rows_read":     rows_read,
    "rows_loaded":   valid.count(),
    "rows_rejected": rejected.count(),
    "revenue":       round(valid.agg(F.sum("line_amount")).first()[0] or 0.0, 2),
}
print(json.dumps(report, indent=2))
notebookutils.notebook.exit(json.dumps(report))